In [1]:
# Albert Medina Familia
# 22-EISN-2-025

import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from PIL import Image

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

dataset_path = "Dataset"
classes = os.listdir(dataset_path)

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

classes = ["Bracelet", "Chains", "Glasses", "Phone", "Rings", "Watches"]

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

data = []
for cls in classes:
    class_path = os.path.join(dataset_path, cls)
    images = os.listdir(class_path)
    for img in images:
        img_path = os.path.join(class_path, img)
        data.append((img_path, cls))

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

from torchvision.datasets import ImageFolder

dataset = ImageFolder(root=dataset_path, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT 
model = resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

import torch.nn as nn

num_classes = len(classes)
model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

from torch.utils.data.dataset import random_split

train_size = int(0.8 * len(data_loader.dataset))
test_size = len(data_loader.dataset) - train_size
train_dataset, test_dataset = random_split(data_loader.dataset, [train_size, test_size])

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dispositivo

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

lr = 1e-3

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

n_epochs = 8

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

perdidastrain = []
perdidasval = []
acuraciatrain = []
acuraciaval = []
error_ratetrain = []
error_rateval = []

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

from tqdm import tqdm

for epoch in range(n_epochs):
    model.train()
    perdida=0
    aciertos=0
    errores=0
    
    for images, labels in tqdm(train_loader):
        
        images, labels = images.to(dispositivo), labels.to(dispositivo)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        perdida+=loss.item()
        _, preds = torch.max(outputs, 1)
        aciertos += (preds == labels).sum().item()
        errores += (preds != labels).sum().item()
        
    perdidastrain.append(perdida/len(train_loader))
    acuraciatrain.append(aciertos/len(train_loader.dataset))
    error_ratetrain.append(errores/len(train_loader.dataset))

    model.eval()
    perdida=0
    aciertos=0
    errores=0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(dispositivo), labels.to(dispositivo)
            outputs = model(images)
            loss = criterion(outputs, labels)
            perdida+=loss.item()
            _, preds = torch.max(outputs, 1)
            aciertos += (preds == labels).sum().item()
            errores += (preds != labels).sum().item()
    perdidasval.append(perdida/len(test_loader))
    acuraciaval.append(aciertos/len(test_loader.dataset))
    error_rateval.append(errores/len(test_loader.dataset))
    
    print(f'Epoch {epoch+1}/{n_epochs}, train loss: {perdidastrain[-1]:.4f}, train accuracy: {acuraciatrain[-1]:.4f}, train error rate: {error_ratetrain[-1]:.4f}, val loss: {perdidasval[-1]:.4f}, val accuracy: {acuraciaval[-1]:.4f}, val error rate: {error_rateval[-1]:.4f}') 


In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

import numpy as np

with torch.no_grad():
    model.eval()
    preds = []
    targets = []
    for images, labels in test_loader:
        images, labels = images.to(dispositivo), labels.to(dispositivo)
        outputs = model(images)
        _, pred = torch.max(outputs, 1)
        preds.append(pred.cpu().numpy())
        targets.append(labels.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

from safetensors.torch import   save_model, load_model
save_model(model, 'detector_accesorios.safetensors')

In [ ]:
# Albert Medina Familia
# 22-EISN-2-025

import torch
from torchvision import transforms
import gradio as gr
from PIL import Image
from safetensors.torch import load_model

clothes = ["Bracelet", "Chains", "Glasses", "Phone", "Rings", "Watches"]

# Cargar el modelo previamente entrenado
model = resnet18(weights=weights)
num_classes = len(clothes)
model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)
load_model(model, 'detector_accesorios.safetensors')

# Función de predicción
def predict(image):
    
    image = image['composite'].convert('RGB')
    image = image.resize((224, 224))
    image = transforms.ToTensor()(image)
    image = image.unsqueeze(0)

    # Realizar la predicción
    with torch.no_grad():
        model.eval()
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

        # Retornar la etiqueta predicha
        return clothes[pred.item()]

# Interfaz de usuario Gradio
gr.Interface(fn=predict,inputs=gr.ImageEditor(type='pil', crop_size="1:1", ) ,outputs="text").launch(share=True)